# Memory Governance — Conflicts, Decay, Audit Trails

Long-term memory without governance becomes **stale, contradictory, and un-auditable**. Production agents need rules for:

- **Conflicts** — "prefers email" vs "prefers phone" — which wins?
- **Decay** — 18-month-old shipping preference may no longer apply
- **Audit trails** — who wrote this memory, when, and from what source?

```
  write request
       │
       ▼
  conflict detect ──► resolve (newer / explicit / confidence)
       │
       ▼
  governed store ──► audit log (append-only)
       │
       ▼
  read request ──► decay filter ──► ranked active memories
```

This notebook implements **ShopCo Governed Memory** — a production-style LTM layer with conflict resolution, TTL decay, and full audit trails. Plain Python, offline by default.

In [ ]:
"""
ShopCo Support — memory governance lab.
"""

import json
import os
import re
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timedelta, timezone
from enum import Enum
from typing import Any, Literal

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)

USE_LIVE_LLM = False


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


def utc_now() -> datetime:
    return datetime.now(timezone.utc)


def parse_ts(ts: str) -> datetime:
    return datetime.fromisoformat(ts.replace("Z", "+00:00"))


POLICY_SNIPPETS = {
    "returns": "Returns within 30 days. [pol-returns-01]",
    "shipping": "Free shipping over $50. [pol-shipping-02]",
}

print("ShopCo governance lab ready.")

---

## 1. Why Governance Matters

| Failure mode | Without governance | With governance |
|--------------|-------------------|-----------------|
| Contradictory prefs | Agent flip-flops answers | Conflict resolution picks winner |
| Stale facts | "Ship to old address" | TTL decay archives expired memories |
| Compliance gap | No proof of what agent "knew" | Audit trail for every write/read |
| Bad extraction | LLM stores wrong fact forever | Confidence + human review queue |

Governance is not optional for regulated support (billing disputes, refund eligibility).

---

## 2. Governed Memory Record Model

In [ ]:
class MemoryStatus(str, Enum):
    ACTIVE = "active"
    SUPERSEDED = "superseded"
    EXPIRED = "expired"
    DELETED = "deleted"


MemoryType = Literal["semantic", "episodic", "procedural"]
MemorySource = Literal["user_explicit", "agent_inferred", "system_import", "human_review"]


@dataclass
class GovernedMemory:
    id: str
    user_id: str
    content: str
    memory_type: MemoryType
    source: MemorySource
    confidence: float
    status: MemoryStatus = MemoryStatus.ACTIVE
    created_at: str = ""
    updated_at: str = ""
    expires_at: str | None = None
    last_accessed_at: str = ""
    superseded_by: str | None = None
    topic: str = "general"
    metadata: dict[str, Any] = field(default_factory=dict)


DEFAULT_TTL_DAYS: dict[MemoryType, int] = {
    "semantic": 365,
    "episodic": 90,
    "procedural": 180,
}


def new_memory(
    user_id: str,
    content: str,
    memory_type: MemoryType = "semantic",
    source: MemorySource = "user_explicit",
    confidence: float = 0.9,
    topic: str = "general",
) -> GovernedMemory:
    now = utc_now()
    ttl = DEFAULT_TTL_DAYS[memory_type]
    return GovernedMemory(
        id=f"mem-{uuid.uuid4().hex[:8]}",
        user_id=user_id,
        content=content,
        memory_type=memory_type,
        source=source,
        confidence=confidence,
        created_at=now.isoformat(),
        updated_at=now.isoformat(),
        expires_at=(now + timedelta(days=ttl)).isoformat(),
        last_accessed_at=now.isoformat(),
        topic=topic,
    )


sample = new_memory("alice", "Prefers email contact", topic="contact_channel")
print(pretty(sample))

---

## 3. Audit Trail — Append-Only Event Log

In [ ]:
AuditAction = Literal[
    "MEMORY_CREATE", "MEMORY_UPDATE", "MEMORY_SUPERSEDE",
    "MEMORY_EXPIRE", "MEMORY_DELETE", "MEMORY_READ", "CONFLICT_DETECTED", "CONFLICT_RESOLVED",
]


@dataclass
class AuditEvent:
    event_id: str
    timestamp: str
    action: AuditAction
    user_id: str
    memory_id: str | None
    actor: str
    details: dict[str, Any] = field(default_factory=dict)


class AuditLog:
    def __init__(self) -> None:
        self._events: list[AuditEvent] = []

    def record(
        self,
        action: AuditAction,
        *,
        user_id: str,
        memory_id: str | None = None,
        actor: str = "system",
        **details: Any,
    ) -> AuditEvent:
        evt = AuditEvent(
            event_id=f"aud-{uuid.uuid4().hex[:8]}",
            timestamp=utc_now().isoformat(),
            action=action,
            user_id=user_id,
            memory_id=memory_id,
            actor=actor,
            details=details,
        )
        self._events.append(evt)
        return evt

    def for_user(self, user_id: str) -> list[AuditEvent]:
        return [e for e in self._events if e.user_id == user_id]

    def for_memory(self, memory_id: str) -> list[AuditEvent]:
        return [e for e in self._events if e.memory_id == memory_id]

    def timeline(self, user_id: str) -> list[str]:
        return [f"{e.timestamp[:19]} {e.action} {e.memory_id or '-'} {e.details}" for e in self.for_user(user_id)]

---

## 4. Conflict Detection

In [ ]:
@dataclass
class MemoryConflict:
    topic: str
    existing: GovernedMemory
    incoming: GovernedMemory
    reason: str


CONFLICT_TOPICS: dict[str, list[tuple[str, str]]] = {
    "contact_channel": [
        (r"email", r"phone|call|sms"),
        (r"phone|call", r"email"),
    ],
    "refund_preference": [
        (r"store credit", r"cash|bank|original payment"),
        (r"cash|refund to card", r"store credit"),
    ],
}


def infer_topic(content: str) -> str:
    c = content.lower()
    if "email" in c or "phone" in c or "contact" in c:
        return "contact_channel"
    if "refund" in c or "store credit" in c:
        return "refund_preference"
    if "concise" in c or "detailed" in c:
        return "response_style"
    return "general"


def detect_conflict(existing: GovernedMemory, incoming: GovernedMemory) -> MemoryConflict | None:
    if existing.topic != incoming.topic or existing.topic == "general":
        if existing.topic == "general":
            incoming.topic = infer_topic(incoming.content)
            existing.topic = infer_topic(existing.content)
        if existing.topic != incoming.topic:
            return None
    pairs = CONFLICT_TOPICS.get(existing.topic, [])
    ex, inc = existing.content.lower(), incoming.content.lower()
    for pat_a, pat_b in pairs:
        if re.search(pat_a, ex) and re.search(pat_b, inc):
            return MemoryConflict(existing.topic, existing, incoming, f"topic={existing.topic} contradictory values")
        if re.search(pat_a, inc) and re.search(pat_b, ex):
            return MemoryConflict(existing.topic, existing, incoming, f"topic={existing.topic} contradictory values")
    if existing.topic == incoming.topic and existing.content.strip().lower() != incoming.content.strip().lower():
        if existing.topic != "general":
            return MemoryConflict(existing.topic, existing, incoming, "same topic, different content")
    return None


old = new_memory("alice", "Prefers email contact", topic="contact_channel")
new = new_memory("alice", "Please call me by phone only", topic="contact_channel", source="user_explicit")
conflict = detect_conflict(old, new)
print(conflict.reason if conflict else "no conflict")

---

## 5. Conflict Resolution Strategies

In [ ]:
class ResolutionStrategy(str, Enum):
    PREFER_NEWER = "prefer_newer"
    PREFER_EXPLICIT = "prefer_explicit"
    PREFER_HIGHER_CONFIDENCE = "prefer_higher_confidence"
    QUEUE_HUMAN_REVIEW = "queue_human_review"


@dataclass
class ResolutionResult:
    winner: GovernedMemory
    loser: GovernedMemory
    strategy: ResolutionStrategy
    needs_review: bool = False


SOURCE_PRIORITY = {"user_explicit": 4, "human_review": 3, "system_import": 2, "agent_inferred": 1}


def resolve_conflict(conflict: MemoryConflict, strategy: ResolutionStrategy) -> ResolutionResult:
    ex, inc = conflict.existing, conflict.incoming
    if strategy == ResolutionStrategy.QUEUE_HUMAN_REVIEW:
        return ResolutionResult(winner=ex, loser=inc, strategy=strategy, needs_review=True)
    if strategy == ResolutionStrategy.PREFER_EXPLICIT:
        ex_pri = SOURCE_PRIORITY.get(ex.source, 0)
        inc_pri = SOURCE_PRIORITY.get(inc.source, 0)
        if inc_pri > ex_pri:
            return ResolutionResult(winner=inc, loser=ex, strategy=strategy)
        if ex_pri > inc_pri:
            return ResolutionResult(winner=ex, loser=inc, strategy=strategy)
    if strategy == ResolutionStrategy.PREFER_HIGHER_CONFIDENCE:
        if inc.confidence > ex.confidence:
            return ResolutionResult(winner=inc, loser=ex, strategy=strategy)
        return ResolutionResult(winner=ex, loser=inc, strategy=strategy)
    # PREFER_NEWER (default)
    inc_ts = parse_ts(inc.created_at)
    ex_ts = parse_ts(ex.created_at)
    if inc_ts >= ex_ts:
        return ResolutionResult(winner=inc, loser=ex, strategy=ResolutionStrategy.PREFER_NEWER)
    return ResolutionResult(winner=ex, loser=inc, strategy=ResolutionStrategy.PREFER_NEWER)


result = resolve_conflict(conflict, ResolutionStrategy.PREFER_EXPLICIT)
print(f"Winner: {result.winner.content[:40]}... strategy={result.strategy.value}")

---

## 6. Decay Policies — TTL and Staleness

In [ ]:
def is_expired(mem: GovernedMemory, now: datetime | None = None) -> bool:
    now = now or utc_now()
    if mem.status != MemoryStatus.ACTIVE:
        return mem.status == MemoryStatus.EXPIRED
    if mem.expires_at and parse_ts(mem.expires_at) < now:
        return True
    return False


def staleness_score(mem: GovernedMemory, now: datetime | None = None) -> float:
    """0.0 = fresh, 1.0 = fully stale."""
    now = now or utc_now()
    if not mem.expires_at:
        return 0.0
    created = parse_ts(mem.created_at)
    expires = parse_ts(mem.expires_at)
    total = (expires - created).total_seconds() or 1
    elapsed = (now - created).total_seconds()
    return min(1.0, max(0.0, elapsed / total))


def apply_decay(mem: GovernedMemory, now: datetime | None = None) -> GovernedMemory:
    now = now or utc_now()
    if is_expired(mem, now):
        mem.status = MemoryStatus.EXPIRED
        mem.updated_at = now.isoformat()
    return mem


old_mem = new_memory("alice", "Returned ORD-8842 in June", memory_type="episodic", topic="returns")
old_mem.created_at = (utc_now() - timedelta(days=120)).isoformat()
old_mem.expires_at = (utc_now() - timedelta(days=5)).isoformat()
print("Expired:", is_expired(old_mem))
print("Staleness:", round(staleness_score(old_mem), 2))

---

## 7. GovernedMemoryStore — Write, Read, Govern

In [ ]:
@dataclass
class WriteResult:
    memory: GovernedMemory
    action: Literal["created", "superseded", "duplicate", "queued_review"]
    conflict: MemoryConflict | None = None


class GovernedMemoryStore:
    def __init__(
        self,
        resolution: ResolutionStrategy = ResolutionStrategy.PREFER_EXPLICIT,
    ) -> None:
        self._memories: dict[str, GovernedMemory] = {}
        self.audit = AuditLog()
        self.resolution = resolution
        self.review_queue: list[MemoryConflict] = []

    def _active_for_user(self, user_id: str) -> list[GovernedMemory]:
        return [m for m in self._memories.values() if m.user_id == user_id and m.status == MemoryStatus.ACTIVE]

    def write(self, mem: GovernedMemory, *, actor: str = "agent") -> WriteResult:
        mem.topic = mem.topic or infer_topic(mem.content)
        for existing in self._active_for_user(mem.user_id):
            conflict = detect_conflict(existing, mem)
            if not conflict:
                continue
            self.audit.record("CONFLICT_DETECTED", user_id=mem.user_id, memory_id=existing.id, actor=actor, topic=mem.topic)
            resolved = resolve_conflict(conflict, self.resolution)
            if resolved.needs_review:
                self.review_queue.append(conflict)
                self.audit.record("CONFLICT_DETECTED", user_id=mem.user_id, actor=actor, queued=True)
                return WriteResult(mem=existing, action="queued_review", conflict=conflict)
            loser = resolved.loser
            winner = resolved.winner
            loser.status = MemoryStatus.SUPERSEDED
            loser.superseded_by = winner.id
            loser.updated_at = utc_now().isoformat()
            self.audit.record("MEMORY_SUPERSEDE", user_id=mem.user_id, memory_id=loser.id, actor=actor, by=winner.id)
            self.audit.record("CONFLICT_RESOLVED", user_id=mem.user_id, memory_id=winner.id, actor=actor, strategy=resolved.strategy.value)
            if winner.id not in self._memories:
                self._memories[winner.id] = winner
                self.audit.record("MEMORY_CREATE", user_id=winner.user_id, memory_id=winner.id, actor=actor)
            return WriteResult(memory=winner, action="superseded", conflict=conflict)

        self._memories[mem.id] = mem
        self.audit.record("MEMORY_CREATE", user_id=mem.user_id, memory_id=mem.id, actor=actor, source=mem.source)
        return WriteResult(memory=mem, action="created")

    def read(self, user_id: str, query: str = "", *, actor: str = "agent") -> list[GovernedMemory]:
        now = utc_now()
        active: list[GovernedMemory] = []
        for mem in list(self._memories.values()):
            if mem.user_id != user_id:
                continue
            apply_decay(mem, now)
            if mem.status == MemoryStatus.EXPIRED:
                self.audit.record("MEMORY_EXPIRE", user_id=user_id, memory_id=mem.id, actor="decay_job")
            if mem.status != MemoryStatus.ACTIVE:
                continue
            mem.last_accessed_at = now.isoformat()
            active.append(mem)
        self.audit.record("MEMORY_READ", user_id=user_id, actor=actor, count=len(active), query=query)
        if query:
            q = query.lower()
            active = [m for m in active if any(w in m.content.lower() for w in q.split())]
        active.sort(key=lambda m: (1 - staleness_score(m, now), m.confidence), reverse=True)
        return active

    def delete(self, memory_id: str, *, actor: str = "admin") -> bool:
        mem = self._memories.get(memory_id)
        if not mem:
            return False
        mem.status = MemoryStatus.DELETED
        mem.updated_at = utc_now().isoformat()
        self.audit.record("MEMORY_DELETE", user_id=mem.user_id, memory_id=memory_id, actor=actor)
        return True

    def history(self, user_id: str) -> list[GovernedMemory]:
        return sorted(
            [m for m in self._memories.values() if m.user_id == user_id],
            key=lambda m: m.created_at,
        )

---

## 8. Demo — Contact Channel Conflict

In [ ]:
store = GovernedMemoryStore(resolution=ResolutionStrategy.PREFER_EXPLICIT)

r1 = store.write(new_memory("alice", "Prefers email contact", topic="contact_channel", source="agent_inferred"))
r2 = store.write(new_memory("alice", "Please use phone only from now on", topic="contact_channel", source="user_explicit"))

print(f"Write 1: {r1.action}, Write 2: {r2.action}")
active = store.read("alice", "contact")
print("Active:", [m.content for m in active])
print("History statuses:", [(m.content[:30], m.status.value) for m in store.history("alice")])

---

## 9. Demo — Episodic Memory Decay

In [ ]:
decay_store = GovernedMemoryStore()
ep = new_memory("bob", "Escalated billing issue for ORD-2001", memory_type="episodic", topic="billing")
ep.created_at = (utc_now() - timedelta(days=100)).isoformat()
ep.expires_at = (utc_now() - timedelta(days=1)).isoformat()
decay_store.write(ep)

before = decay_store.read("bob")
print(f"Active after decay sweep: {len(before)} memories")
expired = [m for m in decay_store.history("bob") if m.status == MemoryStatus.EXPIRED]
print(f"Expired: {len(expired)}")

---

## 10. Demo — Audit Trail Forensics

In [ ]:
print("Alice audit timeline:")
for line in store.audit.timeline("alice"):
    print(f"  {line}")

if active:
    mid = active[0].id
    print(f"\nEvents for memory {mid}:")
    for e in store.audit.for_memory(mid):
        print(f"  {e.action} by {e.actor}")

---

## 11. Human Review Queue

In [ ]:
review_store = GovernedMemoryStore(resolution=ResolutionStrategy.QUEUE_HUMAN_REVIEW)
review_store.write(new_memory("carol", "Prefers email", topic="contact_channel", source="agent_inferred"))
r = review_store.write(new_memory("carol", "Prefers phone", topic="contact_channel", source="agent_inferred"))

print(f"Write result: {r.action}, queue size: {len(review_store.review_queue)}")


def approve_review(store: GovernedMemoryStore, conflict: MemoryConflict, pick: Literal["existing", "incoming"], reviewer: str) -> GovernedMemory:
    winner = conflict.incoming if pick == "incoming" else conflict.existing
    loser = conflict.existing if pick == "incoming" else conflict.incoming
    loser.status = MemoryStatus.SUPERSEDED
    loser.superseded_by = winner.id
    store._memories[winner.id] = winner
    winner.source = "human_review"
    winner.confidence = 1.0
    store.audit.record("CONFLICT_RESOLVED", user_id=winner.user_id, memory_id=winner.id, actor=reviewer, manual=True)
    store.review_queue.remove(conflict)
    return winner


if review_store.review_queue:
    approved = approve_review(review_store, review_store.review_queue[0], "incoming", "reviewer-jane")
    print(f"Approved: {approved.content}")

---

## 12. ShopCo Support Agent with Governed Memory

In [ ]:
class GovernedSupportAgent:
    def __init__(self, store: GovernedMemoryStore) -> None:
        self.store = store

    def handle(self, user_id: str, message: str) -> str:
        q = message.lower()

        if "remember" in q or "prefer" in q:
            source: MemorySource = "user_explicit" if "remember" in q else "agent_inferred"
            mem = new_memory(user_id, message, source=source, topic=infer_topic(message))
            wr = self.store.write(mem, actor="support_agent")
            if wr.action == "queued_review":
                return "I've noted that — a human reviewer will confirm conflicting preferences."
            return "Saved to your governed memory profile."

        if "what do you know" in q or "my preferences" in q:
            memories = self.store.read(user_id, query="preference contact", actor="support_agent")
            if not memories:
                return "I have no active preferences on file."
            lines = [f"- {m.content} (confidence={m.confidence}, topic={m.topic})" for m in memories]
            return "Active memories:\n" + "\n".join(lines)

        if "return" in q:
            return POLICY_SNIPPETS["returns"]
        return "How can I help with your ShopCo order?"


agent = GovernedSupportAgent(store)
print(agent.handle("alice", "What are my contact preferences?"))
print()
print(agent.handle("alice", "Remember I prefer SMS for urgent issues"))

---

## 13. Governance Policy Configuration

In [ ]:
GOVERNANCE_POLICY = {
    "default_resolution": ResolutionStrategy.PREFER_EXPLICIT.value,
    "low_confidence_threshold": 0.6,
    "queue_review_below_confidence": True,
    "ttl_days": DEFAULT_TTL_DAYS,
    "decay_job": "run on every read + nightly batch",
    "audit_retention_days": 2555,  # ~7 years for compliance
    "pii_topics": ["contact_channel", "billing", "address"],
    "require_explicit_for_pii": True,
}

print(pretty(GOVERNANCE_POLICY))

---

## 14. Eval Harness

In [ ]:
eval_store = GovernedMemoryStore(resolution=ResolutionStrategy.PREFER_EXPLICIT)
eval_store.write(new_memory("eve", "Prefers email", topic="contact_channel", source="agent_inferred"))
eval_store.write(new_memory("eve", "Call me on phone", topic="contact_channel", source="user_explicit"))
active_eve = eval_store.read("eve")

EVAL = [
    ("conflict supersedes old", lambda: any(m.status == MemoryStatus.SUPERSEDED for m in eval_store.history("eve"))),
    ("explicit wins", lambda: active_eve and "phone" in active_eve[0].content.lower()),
    ("audit trail exists", lambda: len(eval_store.audit.for_user("eve")) >= 3),
    ("decay marks expired", lambda: any(m.status == MemoryStatus.EXPIRED for m in decay_store.history("bob"))),
    ("review queue works", lambda: len(review_store.review_queue) == 0),  # approved in demo
]

passed = sum(int(fn()) for _, fn in EVAL)
for label, fn in EVAL:
    print(f"{'✓' if fn() else '✗'} {label}")
print(f"\nEval: {passed}/{len(EVAL)}")

---

## 15. Resolution Strategy Comparison

| Strategy | Best for | Risk |
|----------|----------|------|
| **PREFER_NEWER** | Fast-moving prefs | Newer inference may be wrong |
| **PREFER_EXPLICIT** | Support / compliance | Needs source tagging |
| **PREFER_HIGHER_CONFIDENCE** | Noisy extraction | Confidence scores may be miscalibrated |
| **QUEUE_HUMAN_REVIEW** | PII, billing, legal | Slower, needs ops queue |

---

## 16. Common Mistakes

| Mistake | Symptom | Fix |
|---------|---------|-----|
| No conflict detection | Contradictory answers | Topic tagging + pairs |
| Infinite memory TTL | Stale address from 2023 | Per-type TTL decay |
| Mutable audit log | Cannot prove compliance | Append-only `AuditLog` |
| Auto-resolve PII conflicts | Wrong contact channel | Human review queue |
| No `source` field | Can't prefer explicit | Tag user_explicit vs inferred |
| Decay only in batch | Stale reads between jobs | Decay on read + nightly |

---

## 17. Production Checklist

- [ ] `GovernedMemory` schema with status, source, confidence, TTL
- [ ] Conflict detection by topic
- [ ] Resolution strategy per topic (PII → human review)
- [ ] Decay on read + scheduled expiry job
- [ ] Append-only audit log with actor and timestamp
- [ ] Audit retention policy (compliance-driven)
- [ ] Human review queue for unresolved conflicts
- [ ] Memory read/write metrics and alerts

---

## 18. Optional Live LLM — Conflict Summarization for Reviewers

In [ ]:
def summarize_conflict_for_reviewer(conflict: MemoryConflict) -> str:
    if not USE_LIVE_LLM:
        return (
            f"CONFLICT [{conflict.topic}]:\n"
            f"  Existing ({conflict.existing.source}): {conflict.existing.content}\n"
            f"  Incoming ({conflict.incoming.source}): {conflict.incoming.content}\n"
            f"  Reason: {conflict.reason}"
        )

    from langchain_openai import ChatOpenAI

    llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)
    prompt = (
        f"Summarize this memory conflict for a human reviewer. Be concise.\n"
        f"Topic: {conflict.topic}\n"
        f"Existing: {conflict.existing.content}\n"
        f"Incoming: {conflict.incoming.content}"
    )
    return llm.invoke(prompt).content


if conflict:
    print(summarize_conflict_for_reviewer(conflict))

---

## 19. Quiz

1. What is the difference between `SUPERSEDED` and `EXPIRED` memory status?
2. When should you use `QUEUE_HUMAN_REVIEW` instead of `PREFER_NEWER`?
3. Why run decay on read *and* in a nightly job?
4. What must an audit trail record for compliance?
5. Why tag `source` as `user_explicit` vs `agent_inferred`?

---

## 20. Summary

**Memory governance** makes long-term memory trustworthy:

1. **Conflicts** — detect by topic, resolve with explicit > inferred > newer
2. **Decay** — per-type TTL, staleness scoring, expire on read
3. **Audit** — append-only log of every create, supersede, expire, read, delete
4. **Human review** — queue for PII and low-confidence conflicts

ShopCo's `GovernedMemoryStore` wraps any LTM backend (LangMem, Mem0, BaseStore) with these policies. Without governance, memory architecture choice alone is not production-ready.